# **Load Libraries**

In [1]:
from unsloth import FastLanguageModel

from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

from typing import List, Dict, Optional, Any
from tqdm.auto import tqdm

import re

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Load Dataset**

In [2]:
def load_and_merge_datasets(
    dataset_configs: List[Dict[str, Any]], 
    split: Optional[str] = None
) -> Dataset | DatasetDict:
    if not dataset_configs:
        raise ValueError("`dataset_configs` must contain at least one dataset.")
    
    loaded = []
    for config in tqdm(dataset_configs, desc='INFO: Loading Dataset'):
        config = config.copy()
        path = config.pop("path")
        effc_split = config.pop("split", split)

        try:
            ds = load_dataset(path, split=effc_split, **config)
            loaded.append(ds)

            print(f'Loaded dataset: {config} !')
        except:
            print(f'Error loading the dataset: {config}')

    
    if len(loaded) == 1:
        return loaded[0]
    
    if all(isinstance(ds, Dataset) for ds in loaded):
        return concatenate_datasets(loaded)

In [3]:
ds = load_and_merge_datasets(
    [
        {'path': 'arm-team/Stage2_RL_gsm8k'},
        {'path': 'arm-team/Stage2_RL_MATH'},
        {'path': 'arm-team/Stage2_RL_csqa'},
    ],
    split='train'
)

INFO: Loading Dataset:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded dataset: {} !
Loaded dataset: {} !
Loaded dataset: {} !


# **Find token length & pre-process**

In [19]:
model_name = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 4096

In [ ]:
# Load Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False
)

# Define & add special tokens to tokenizer
format_tag = [
    '<ANSWER>',   '</ANSWER>',
    '<CODE>',     '</CODE>',
    '<COT>',      '</COT>',
    '<LONG_COT>', '<LONG_COT>'
]
tokenizer.add_tokens(format_tag, special_tokens=True)

In [26]:
def tokenize(example: Dict[str, str]) -> Dict[str, str | int]:
    instruction = example['question']
    solution = example['answer']

    message = [
        {'role': 'user',      'content': instruction},
        # {'role': 'assistant', 'content': solution   }
    ]

    msg_tokenized = tokenizer.apply_chat_template(
        message,
        tokenize=True,
        add_generation_prompt=True
    )['input_ids']

    example['Token_Length'] = len(msg_tokenized)
    return example

In [27]:
# Assign token length based on output variables
ds_full = ds.map(tokenize)

# Transform into pandas Dataframe format
df = ds_full.to_pandas()

Map:   0%|          | 0/19800 [00:00<?, ? examples/s]

In [34]:
df[df['Token_Length'] > 512]

,question,answer,Token_Length
7449,A portion of the graph of a quadratic function...,$\boxed{21}$,740
7566,"The graph of $y=f(x)$ is shown below, with $1$...",$\boxed{-7}$,729
7633,The entire graph of the function $f(x)$ is sho...,$\boxed{3}$,536
7730,A portion of the graph of $y = f(x)$ is shown ...,$\boxed{-8}$,703
7788,"The graphs of two functions, $p(x)$ and $q(x),...",$\boxed{-10}$,830
...,...,...,...
14007,The graph shows the birth month of 100 famous ...,$\boxed{8}$,1052
14076,Sandy's daughter has a playhouse in the back y...,$\boxed{ \$ 54.60}$,631
14490,Parallelepiped $ABCDEFGH$ is generated by vect...,$\boxed{4}$,563
14506,"A unit cube has vertices $P_1,P_2,P_3,P_4,P_1'...",$\boxed{\frac{3 \sqrt{2}}{4}}$,754
